# Übungsblatt Web-API, JSON

## Aufgabe
**1.**
Gehe auf die Website https://fdc.nal.usda.gov/api-key-signup und registriere dich.
Danach erhältst du einen API-Key.

Mit diesem API-Key kannst du die USDA FoodData Central API nutzen, um Nährwertangaben zu Lebensmitteln abzufragen.
Die API-Dokumentation findest du hier: https://fdc.nal.usda.gov/api-guide.
Die Dukumentation der FoodData Central Foundation Foods Documentation findest du hier: https://fdc.nal.usda.gov/Foundation_Foods_Documentation.

**2.**

In der Vertiefung haben wir die Nährwertangaben für eine Zutat abgefragt und eine Liste aller Nährwerte erhalten.
Mit dem Endpunkt /foods kannst du die Nährwertangaben für mehrere Lebensmittel gleichzeitig abfragen. Ausserdem kannst du über den Parameter nutrients schon bei der Anfrage festlegen, welche Nährstoffe zurückgegeben werden. Der Endpunkt unterstützt dabei bis zu 20 FDC-IDs und bis zu 25 Nährstoffnummern.

**3.**

Der Endpunkt zum Suchen von fdcId-Werten ist:
https://api.nal.usda.gov/fdc/v1/foods/search

Dort kannst du die Zutat eingeben, für die du Nährwertangaben abfragen möchtest.
In der Vertiefung haben wir nur den ersten Treffer verwendet. Der Endpunkt liefert aber mehrere Treffer zurück.

Schreibe ein Programm, das die Nährwertangaben für die ersten 5 Treffer abfragt und für alle 5 Treffer ausgibt.
Der Search-Endpunkt liefert Trefferlisten; die Seitenzahl ist über pageSize steuerbar.
Suche nach dem Lebensmittel "potato" und gib die Nährwertangaben für die ersten 5 Treffer aus.

**4.**

Es sollen nicht alle Nährwertangaben ausgegeben werden, sondern nur:

- Kalorien
- Kohlenhydrate
- Fett
- Eiweiss

Schreibe ein Programm, das die Nährwertangaben für die ersten 5 Treffer abfragt und für die gefundenen Lebensmittel nur die Kalorien, Kohlenhydrate, Fett und Eiweiss ausgibt.
Bei der Abfrage der Nährwertangaben kannst du über den Parameter nutrients schon festlegen, dass nur die Nährwerte für Kalorien, Kohlenhydrate, Fett und Eiweiss zurückgegeben werden.
Die Liste der Nährstoffnummern findest du hier: https://fdc.nal.usda.gov/download-datasets. Das File 'nutrients.csv' stammt aus dem Download-Verzeichnis und enthält die Nährstoffnummern für die gesuchten Nährstoffe:

- Kalorien (Atwater General Factors): 957
- Kohlenhydrate: 203
- Fett: 204
- Eiweiss: 205

**5.**

Vergleiche die Nährwertangaben von allen 5 Treffern miteinander. Welche Zutat hat die meisten Kalorien? Welche Zutat hat die meisten Kohlenhydrate? Welche Zutat hat die meisten Fett? Welche Zutat hat die meisten Eiweiss?

In [35]:
import requests

API_KEY = "UNU9QhAFgJDtGtzdMZDpff5e1FIaDEtbGSTbCoh2"
SEARCH_URL = "https://api.nal.usda.gov/fdc/v1/foods/search"
DETAIL_URL = "https://api.nal.usda.gov/fdc/v1/foods"


def suche_lebensmittel(suchbegriff, anzahl=5):
    response = requests.post(
        SEARCH_URL,
        params={"api_key": API_KEY},
        json={"query": suchbegriff, "dataType": ["Foundation"], "pageSize": anzahl},
    )
    response.raise_for_status()
    daten = response.json()
    return daten.get("foods", [])


def hole_details(fdc_ids, nutrients=None):
    if nutrients is None:
        nutrients = [957, 203, 204, 205]

    response = requests.post(
        DETAIL_URL,
        params={"api_key": API_KEY},
        json={"fdcIds": fdc_ids, "format": "abridged", "nutrients": nutrients},
    )
    response.raise_for_status()
    return response.json()


def drucke_alle_naehrwerte(foods):
    for i, food in enumerate(foods, start=1):
        print(f"\nTreffer {i}: {food.get('description', 'Unbekannt')}")
        print(f"FDC-ID: {food.get('fdcId')}")
        print(f"Datentyp: {food.get('dataType', 'Unbekannt')}")
        print("Nährwerte:")

        for eintrag in food.get("foodNutrients", []):
            name = eintrag.get("name", "Unbekannt")
            unit = eintrag.get("unitName", "")
            amount = eintrag.get("amount")

            if amount is not None:
                print(f"  - {name}: {amount} {unit}")


suchbegriff = "potato"

suchtreffer = suche_lebensmittel(suchbegriff, anzahl=5)

if not suchtreffer:
    print("Keine Treffer gefunden.")
else:
    fdc_ids = [food["fdcId"] for food in suchtreffer[:5]]
    details = hole_details(fdc_ids)

In [36]:
# in eine dataframe umwandeln
import pandas as pd

df = pd.DataFrame(details)
df.head()
foodNutrients = df["foodNutrients"].explode().apply(pd.Series)
foodNutrients.head()
df_nutrients = pd.concat([df, foodNutrients], axis=1)
df_nutrients

,fdcId,description,dataType,publicationDate,ndbNumber,foodNutrients,number,name,amount,unitName,derivationCode,derivationDescription
0,2261422,"Flour, potato",Foundation,2022-04-28,11413,"[{'number': '204', 'name': 'Total lipid (fat)'...",204,Total lipid (fat),0.951,G,A,Analytical
0,2261422,"Flour, potato",Foundation,2022-04-28,11413,"[{'number': '204', 'name': 'Total lipid (fat)'...",205,"Carbohydrate, by difference",79.900,G,NC,Calculated
0,2261422,"Flour, potato",Foundation,2022-04-28,11413,"[{'number': '204', 'name': 'Total lipid (fat)'...",203,Protein,8.110,G,NC,Calculated
0,2261422,"Flour, potato",Foundation,2022-04-28,11413,"[{'number': '204', 'name': 'Total lipid (fat)'...",957,Energy (Atwater General Factors),361.000,KCAL,NC,Calculated
1,2346403,"Potatoes, gold, without skin, raw",Foundation,2022-10-28,100285,"[{'number': '204', 'name': 'Total lipid (fat)'...",204,Total lipid (fat),0.264,G,A,Analytical
1,2346403,"Potatoes, gold, without skin, raw",Foundation,2022-10-28,100285,"[{'number': '204', 'name': 'Total lipid (fat)'...",957,Energy (Atwater General Factors),73.500,KCAL,NC,Calculated
1,2346403,"Potatoes, gold, without skin, raw",Foundation,2022-10-28,100285,"[{'number': '204', 'name': 'Total lipid (fat)'...",203,Protein,1.810,G,NC,Calculated
1,2346403,"Potatoes, gold, without skin, raw",Foundation,2022-10-28,100285,"[{'number': '204', 'name': 'Total lipid (fat)'...",205,"Carbohydrate, by difference",16.000,G,NC,Calculated
2,2346402,"Potatoes, red, without skin, raw",Foundation,2022-10-28,100284,"[{'number': '204', 'name': 'Total lipid (fat)'...",204,Total lipid (fat),0.248,G,A,Analytical
2,2346402,"Potatoes, red, without skin, raw",Foundation,2022-10-28,100284,"[{'number': '204', 'name': 'Total lipid (fat)'...",203,Protein,2.060,G,NC,Calculated


In [37]:
# Vergleiche die Nährwertangaben von allen 5 Treffern miteinander.
#
# Welche Zutat hat die meisten Kalorien?
df_nutrients[df_nutrients["name"] == "Energy (Atwater General Factors)"].sort_values(
    "amount", ascending=False
)[["description", "amount", "unitName"]].head(5)

,description,amount,unitName
0,"Flour, potato",361.0,KCAL
3,"Potatoes, russet, without skin, raw",83.4,KCAL
4,"Sweet potatoes, orange flesh, without skin, raw",79.0,KCAL
2,"Potatoes, red, without skin, raw",75.6,KCAL
1,"Potatoes, gold, without skin, raw",73.5,KCAL


In [38]:
# Welche Zutat hat die meisten Kohlenhydrate?
df_nutrients[df_nutrients["name"] == "Carbohydrate, by difference"].sort_values(
    "amount", ascending=False
)[["description", "amount", "unitName"]].head(5)

,description,amount,unitName
0,"Flour, potato",79.9,G
3,"Potatoes, russet, without skin, raw",17.8,G
4,"Sweet potatoes, orange flesh, without skin, raw",17.3,G
2,"Potatoes, red, without skin, raw",16.3,G
1,"Potatoes, gold, without skin, raw",16.0,G


In [39]:
# Welche Zutat hat die meisten Fett?
df_nutrients[df_nutrients["name"] == "Total lipid (fat)"].sort_values(
    "amount", ascending=False
)[["description", "amount", "unitName"]].head(5)

,description,amount,unitName
0,"Flour, potato",0.951,G
4,"Sweet potatoes, orange flesh, without skin, raw",0.375,G
3,"Potatoes, russet, without skin, raw",0.360,G
1,"Potatoes, gold, without skin, raw",0.264,G
2,"Potatoes, red, without skin, raw",0.248,G


In [40]:
# Welche Zutat hat die meisten Eiweiss?
df_nutrients[df_nutrients["name"] == "Protein"].sort_values("amount", ascending=False)[
    ["description", "amount", "unitName"]
].head(5)

,description,amount,unitName
0,"Flour, potato",8.11,G
3,"Potatoes, russet, without skin, raw",2.27,G
2,"Potatoes, red, without skin, raw",2.06,G
1,"Potatoes, gold, without skin, raw",1.81,G
4,"Sweet potatoes, orange flesh, without skin, raw",1.58,G
